# PMT Quality Selection

Carga desde el cache de preprocessing, aplica los filtros de calidad robustos
y un filtro adicional de amplitud, produciendo `wfset` listo para el análisis.

**Prerequisito:** `pmt_preprocess.py --run <RUN>`

| Corte | Descripción | Razón |
|---|---|---|
| `baseline_nmad` | Baseline muy desplazada | `baseline_shift` |
| `pre_rms_nmad` | Ruido elevado en baseline | `baseline_rms_high` |
| `pre_max_nmad` | Pico espurio antes del pulso | `pretrigger_peak` |
| `pre_integral_nmad` | Exceso de carga antes del trigger | `pretrigger_charge` |
| `peak_amp_nmad` | Amplitud anómala | `peak_amp_outlier` |
| `charge_nmad` | Carga integrada anómala | `charge_outlier` |
| `peak_tick_abs` | Pico desplazado en tiempo | `peak_time_shift` |
| `peak_amp_abs_min/max` | Amplitud fuera de rango | `peak_amp_low/high` |
| `adc_min_threshold` | Saturación negativa | `adc_negative_saturation` |

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.input_output.pickle_hdf5_reader import WaveformSet_from_hdf5_pickle

from quality_filter import (
    DEFAULT_QUALITY_WINDOWS,
    DEFAULT_QUALITY_CUTS,
    build_wfset_quality,
)

In [ ]:
# --- Configuración del run ---
run     = 43363
dettype = 'pmt'

cache_file = f"cache/wfset_run{run:06d}_{dettype}_ana.hdf5"
wfset_triggered_all = WaveformSet_from_hdf5_pickle(cache_file)
print(f"Cargadas {len(wfset_triggered_all.waveforms)} waveforms desde {cache_file}")

## Configuración de cortes de calidad

Modifica los valores aquí para afinar los filtros. El resto del notebook no cambia.

In [ ]:
# Ventanas en ticks — ajusta según los diagnósticos del notebook 01.
quality_windows = {
    **DEFAULT_QUALITY_WINDOWS,
    # "signal": slice(55, 105),  # ejemplo: adelantar la ventana de señal
}

# Cortes — modifica solo los que necesites; el resto toma el valor por defecto.
quality_cuts = {
    **DEFAULT_QUALITY_CUTS,
    # "peak_amp_abs_min": 500.0,
}

## Aplicar filtros de calidad

In [ ]:
(
    wfset_quality,
    quality_stats_by_channel,
    quality_pass_by_wf_id,
    quality_reasons_by_wf_id,
    quality_reason_counts,
    quality_channel_summary,
) = build_wfset_quality(wfset_triggered_all, cuts=quality_cuts, windows=quality_windows)

## Diagnósticos por canal

Cambia `target_quality_endpoint` y `target_quality_channel` para inspeccionar un canal concreto.

In [ ]:
target_quality_endpoint = 110
target_quality_channel  = 14
metric_to_plot          = "peak_amp"
# Métricas disponibles: baseline, baseline_rms, pre_max, pre_integral_pos,
#                       peak_tick, peak_amp, charge, raw_adc_min, raw_adc_max

channel_key  = (target_quality_endpoint, target_quality_channel)
ch_summary   = quality_channel_summary.get(channel_key, {})
total        = ch_summary.get("total", 0)
n_pass       = ch_summary.get("pass",  0)
n_reject     = ch_summary.get("reject", 0)

print(f"Canal {channel_key}: {total} waveforms")
print(f"  aceptadas:  {n_pass}")
print(f"  rechazadas: {n_reject}")

# Razones de rechazo en este canal
reasons_this_channel = Counter()
for wf_id, reasons in quality_reasons_by_wf_id.items():
    # Solo contar si el wf pertenece al canal (necesitamos el endpoint+channel)
    pass  # accedemos directo al resumen global filtrado abajo

# Resumen global de razones de rechazo
fig, ax = plt.subplots(figsize=(9, 4))
if quality_reason_counts:
    reasons, counts = zip(*quality_reason_counts.most_common())
    ax.bar(range(len(reasons)), counts)
    ax.set_xticks(range(len(reasons)))
    ax.set_xticklabels(reasons, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("Waveforms rechazadas")
    ax.set_title("Razones de rechazo (todos los canales)")
    ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Filtro adicional de amplitud

Filtro conservador sobre la zona del pico. Ajusta `slice_min` y los umbrales según el canal.

In [ ]:
def select_amp(waveform: Waveform) -> bool:
    slice_min = slice(78, 83) if waveform.channel == 16 else slice(75, 80)
    baseline  = waveform.analyses['std'].result['baseline']
    y         = waveform.adcs - baseline
    return np.min(y[slice_min]) > 400 and np.max(y[60:80]) < 7e3

wfset = WaveformSet.from_filtered_WaveformSet(wfset_quality, select_amp, show_progress=True)
wfch  = ChannelWsGrid.clusterize_waveform_set(wfset)
print(f"wfset final: {len(wfset.waveforms)} waveforms")
wfch